# Transformers y Foundation Models en Series Temporales

## Sesión 9: ¿Es la Atención todo lo que necesitas?

En 2017, un grupo de ingenieros de Google publicó un paper con un título que parecía una declaración de guerra: *"Attention is All You Need"*. En él, proponían los **Transformers**: una arquitectura basada en un mecanismo de "atención" que permitía a cada elemento de una secuencia consultar directamente a cualquier otro elemento, sin importar la distancia que los separara. La revolución fue total. En pocos años, los Transformers desbancaron a las LSTMs en traducción automática, generación de texto (GPT, LLaMA) y visión por computador (ViT).

Era cuestión de tiempo que alguien los aplicara al forecasting de series temporales, prometiendo que la atención resolvería los problemas de las LSTMs y las TCNs de un plumazo.

Sin embargo, en 2023, la comunidad científica recibió un baño de agua fría. Un equipo publicó *"Are Transformers Effective for Time Series Forecasting?"*, demostrando que un modelo ridículamente simple —dos capas lineales aplicadas a la serie descompuesta— conseguía **superar sistemáticamente a los Transformers más sofisticados del momento** en benchmarks estándar.

Esta sesión es una invitación a pensar críticamente sobre el *hype* tecnológico y a entender *cuándo* la complejidad de un Transformer está justificada y cuándo es simplemente un sobrecoste que no merece la pena pagar.

---

## Objetivos

Al terminar esta sesión deberías ser capaz de:

- **Comprender** el mecanismo de atención (*Self-Attention*) a nivel conceptual y por qué es más complicado aplicarlo al tiempo que al lenguaje.
- **Identificar** el problema del "tokenismo punto a punto" y por qué destruye la efectividad de la atención en series temporales.
- **Explicar** la solución de PatchTST: dividir la serie en parches antes de aplicar atención.
- **Implementar** el modelo de referencia DLinear (el modelo lineal que puso en jaque a los Transformers).
- **Conocer** la nueva frontera de los Foundation Models (TimeGPT, Lag-Llama) y sus limitaciones prácticas.
- **Razonar** cuándo tiene sentido la complejidad de un Transformer y cuándo es simplemente ruido caro.

## Requisitos técnicos

In [ ]:
import sys
import os

os.environ["USE_TF"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["LIGHTNING_DISABLE_TIPS"] = "1"

sys.path.append("src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl

from sklearn.metrics import mean_absolute_error
from series_temporales import generar_consumo_electrico

# Opcional: para usar TimeGPT (requiere API key gratuita en nixtla.io)
# from nixtla import NixtlaClient


class TimeSeriesDataset(Dataset):
    def __init__(self, df, target_col, window_size=168, forecast_horizon=1):
        self.features = df.drop(columns=[target_col]).values.astype(np.float32)
        self.target = df[target_col].values.astype(np.float32)
        self.window_size = window_size
        self.forecast_horizon = forecast_horizon
        self.length = len(df) - window_size - forecast_horizon + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        X = self.features[idx : idx + self.window_size]
        target_idx = idx + self.window_size + self.forecast_horizon - 1
        y = self.target[target_idx]
        return torch.from_numpy(X), torch.tensor(y)

---

## 1. La Intuición de la Atención

Antes de criticar algo, hay que entenderlo bien. El mecanismo de *Self-Attention* es genuinamente brillante.

Imagina que tienes una secuencia de palabras: *"El banco del río tenía mucho barro"*. Para entender el significado de "banco", necesitas saber si está cerca de "río" (banco de sedimentos) o de "dinero" (banco financiero). La atención permite que la representación de "banco" se actualice **consultando directamente a "río"**, sin importar cuántas palabras haya entre ellos.

En términos matemáticos, cada elemento de la secuencia genera tres vectores: una **Query** (Q, "¿qué busco?"), una **Key** (K, "¿qué ofrezco?") y un **Value** (V, "¿qué información tengo?"). La atención es simplemente:

```text
Atención(Q, K, V) = softmax(Q · Kᵀ / √d) · V
```

El término `Q · Kᵀ` produce una matriz de similitud entre todos los pares de elementos, que determina cuánta "atención" presta cada elemento a los demás.

**El problema:** esa matriz de similitud es de tamaño `[L × L]`, donde `L` es la longitud de la secuencia. Si `L = 168` horas (una semana), la matriz tiene `168 × 168 = 28.224` valores. Si `L = 8.760` horas (un año), tiene **76,7 millones de valores**. El coste computacional crece cuadráticamente: $O(L^2)$.

---

## 2. El Problema del Transformer con el Tiempo

Ahora viene la crítica de fondo. En el lenguaje, cada token (palabra) tiene un **significado semántico** rico e independiente del contexto inmediato. La palabra "banco" contiene en sí misma una ambigüedad que la atención puede resolver consultando las palabras cercanas.

En series temporales, ¿qué significa el token `1.85 kWh` a las `14:00`? **Nada por sí solo.** Su relevancia solo existe en relación a la tendencia que lo rodea: ¿es un pico dentro de una tarde baja? ¿Es el valor habitual de ese martes? Un único punto flotante no tiene "semántica".

Cuando aplicamos atención punto a punto a una serie temporal, estamos calculando la similitud entre valores numéricos aislados. Eso genera una matrix de atención llena de ruido, donde ningún valor individual aporta información suficientemente rica para que la consulta Q-K tenga sentido.

> **La paradoja:** El mecanismo más potente del mundo del NLP es precisamente el menos adecuado para el objeto más secuencial que existe: el tiempo.

Este es el argumento central del paper *"Are Transformers Effective for Time Series Forecasting?"* (Zeng et al., 2023). Sus autores propusieron que el éxito aparente de los Transformers en time series benchmarks se debía a la permutación de las series temporales (*reversible normalization*), no al mecanismo de atención en sí. Demostraron que un modelo llamado **DLinear** —que simplemente descompone la serie en tendencia y residuo y aplica una capa lineal a cada parte— igualaba o superaba a modelos como Autoformer, FEDformer o Informer con una fracción del coste computacional.

---

## 3. El Modelo de Referencia: DLinear

Vamos a implementar DLinear, el modelo que sacudió los cimientos de la comunidad de Transformers en 2023:

In [ ]:
class MovingAvgDecomposition(nn.Module):
    """Descomposición en Tendencia + Residuo mediante media móvil."""
    def __init__(self, kernel_size=25):
        super().__init__()
        # Padding para mantener la longitud de la secuencia
        self.avg = nn.AvgPool1d(kernel_size, stride=1, padding=kernel_size // 2)

    def forward(self, x):
        # x: [Batch, Sequence, Features]
        # AvgPool1d espera [Batch, Features, Sequence]
        x_t = x.permute(0, 2, 1)
        trend = self.avg(x_t)
        # Recortamos 1 elemento extra que aparece por el padding par
        if trend.shape[-1] != x_t.shape[-1]:
            trend = trend[:, :, :-1]
        seasonal = x_t - trend
        return trend.permute(0, 2, 1), seasonal.permute(0, 2, 1)


class DLinear(pl.LightningModule):
    """
    DLinear: Una capa lineal para la tendencia y otra para el residuo.
    Simple. Rápido. Sorprendentemente competitivo.
    """
    def __init__(self, seq_len, pred_len, n_features, kernel_size=25, lr=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.decomp = MovingAvgDecomposition(kernel_size)

        # Una capa lineal por cada componente, por cada feature
        self.linear_trend    = nn.Linear(seq_len, pred_len)
        self.linear_seasonal = nn.Linear(seq_len, pred_len)

    def forward(self, x):
        # x: [Batch, Sequence, Features]
        trend, seasonal = self.decomp(x)

        # Aplicamos la lineal sobre el eje temporal para cada feature
        # -> [Batch, pred_len, Features]
        out_trend    = self.linear_trend(trend.permute(0, 2, 1)).permute(0, 2, 1)
        out_seasonal = self.linear_seasonal(seasonal.permute(0, 2, 1)).permute(0, 2, 1)

        return (out_trend + out_seasonal)[:, -1, 0]  # Devolvemos la predicción del último paso

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = nn.MSELoss()(self(x), y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        loss = nn.MSELoss()(self(x), y)
        self.log("val_loss", loss, prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

---

## 4. La Solución Razonada: PatchTST

Si el problema de los Transformers es que cada punto individual no tiene semántica suficiente, la solución es obvia: **deja de tratar puntos individuales y empieza a tratar segmentos de la serie**.

**PatchTST** (Nie et al., 2023) adaptó la idea de los Vision Transformers (ViT), que dividen una imagen en parches (16×16 píxeles) antes de aplicar atención. Aplicada al tiempo:

```text
Serie original:    [h1, h2, h3, h4, h5, h6, h7, h8, h9, h10, h11, h12]
                    ↓           ↓           ↓           ↓
Parches (P=3):   [h1,h2,h3] [h4,h5,h6] [h7,h8,h9] [h10,h11,h12]
                    Token 1     Token 2     Token 3     Token 4
```

Cada "token" ahora es un segmento de 3 horas consecutivas. Ese segmento sí tiene estructura local: tiene una tendencia, una media, una forma. La atención entre tokens ya tiene sentido: "el segmento de las 7-9 AM se parece mucho al segmento de las 19-21 PM en términos de actividad".

**Beneficios de PatchTST:**
1. **Reducción dramática del coste:** Si la serie tiene `L=336` puntos y el parche tiene `P=16`, en lugar de `336 × 336 = 112.896` cálculos de atención, solo necesitamos `21 × 21 = 441`.
2. **Contexto local incorporado:** Cada token ya contiene información sobre la forma local de la serie.
3. **Longer lookback:** Al tener menos tokens, podemos ampliar la ventana de contexto sin explotar el coste computacional.

In [ ]:
def crear_parches(x, patch_len=16, stride=8):
    """
    Divide una secuencia temporal en parches superpuestos.

    Args:
        x: Tensor de shape [Batch, Sequence, Features]
        patch_len: Longitud de cada parche
        stride: Desplazamiento entre parches (stride < patch_len => parches superpuestos)

    Returns:
        Tensor de shape [Batch, num_patches, patch_len * Features]
    """
    batch_size, seq_len, n_features = x.shape
    parches = []
    for start in range(0, seq_len - patch_len + 1, stride):
        parche = x[:, start : start + patch_len, :]   # [B, patch_len, Features]
        parches.append(parche.reshape(batch_size, -1))  # [B, patch_len * Features]
    return torch.stack(parches, dim=1)  # [B, num_patches, patch_len * Features]

> **Pregunta para discutir:** Con `seq_len=168` (una semana horaria), `patch_len=24` (un día) y `stride=24` (sin solapamiento), ¿cuántos tokens genera la operación de parcheo? ¿Tiene sentido semántico que cada "token" sea un día completo de actividad? ¿Cómo cambia la respuesta si usas `stride=12` (parches solapados)?

---

## 5. El Amanecer de los Foundation Models

La siguiente frontera no es solo una mejor arquitectura, sino **modelos pre-entrenados con billones de puntos de series temporales** procedentes de los dominios más variados: meteorología, bolsa, energía, retail, tráfico web, señales biomédicas...

La promesa es idéntica a la de GPT en el texto: un modelo que ha "leído" toda la historia del tiempo puede predecir cualquier nueva serie con *zero-shot* (sin entrenamiento adicional).

### TimeGPT (Nixtla)
El primer modelo fundacional comercial de propósito general para forecasting. Solo hay que pasarle el DataFrame con la serie y devuelve la predicción sin necesidad de entrenar nada.

In [ ]:
# Ejemplo conceptual (requiere API key gratuita en nixtla.io)
# from nixtla import NixtlaClient
# client = NixtlaClient(api_key="tu_api_key")
# prediccion = client.forecast(df=df, h=24, freq="H", time_col="timestamp", target_col="consumo_kwh")

### Lag-Llama
Basado en la arquitectura Llama (Meta), fue pre-entrenado con cientos de datasets de series temporales de código abierto. Su nombre viene de su especialidad: detectar automáticamente los *lags* (retardos) relevantes de cualquier serie sin necesidad de Feature Engineering manual.

### Moirai (Salesforce) y Chronos (Amazon)
Iniciativas de empresas tecnológicas grandes para crear modelos de forecasting de propósito general que puedan desplegarse en local, sin depender de una API externa.

**Las limitaciones actuales de los Foundation Models:**
1. **Coste computacional:** Modelos grandes requieren hardware potente para inferencia.
2. **Caja negra extrema:** Imposible explicar las predicciones a un cliente.
3. **Bajo rendimiento en series muy específicas:** Un modelo que ha visto datos de bolsa de todo el mundo puede predecir mal el consumo de agua de una pequeña población rural española con patrones muy locales.
4. **Dependencia de APIs externas:** Si Nixtla cierra su servicio, tus predicciones se detienen.

> [!IMPORTANT]
> **El panorama cambia muy rápido.** El paper de DLinear fue publicado en 2022. PatchTST llegó en 2023. TimeGPT se lanzó en 2023. En 2024 han aparecido al menos 5 nuevos modelos "state of the art". La habilidad más valiosa no es conocer el modelo más nuevo, sino entender los principios fundamentales que permiten evaluar críticamente cada nueva propuesta.

---

## 6. El Gran Experimento: DLinear vs Todo

Diseñamos un experimento justo donde comparamos **en los mismos datos y con la misma separación temporal** los modelos de los que disponemos:

In [ ]:
# Preparamos los datos comunes
df = generar_consumo_electrico(fecha_inicio="2026-01-01", periodos=24*180, frecuencia="h",
                               semilla=42, incluir_meteorologia=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp").asfreq("h")
feature_cols = ["consumo_kwh", "temperatura", "hora", "dia_semana"]
df_ml = df[feature_cols].dropna()

n = len(df_ml)
train_df = df_ml.iloc[:int(n * 0.75)]
val_df   = df_ml.iloc[int(n * 0.75):]

SEQ_LEN = 168  # Una semana de contexto
train_ds = TimeSeriesDataset(train_df, "consumo_kwh", window_size=SEQ_LEN)
val_ds   = TimeSeriesDataset(val_df,   "consumo_kwh", window_size=SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False)


def evaluar_modelo(modelo, loader):
    """Evalúa un LightningModule sobre un DataLoader y devuelve el MAE."""
    modelo.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            preds.extend(modelo(X_batch).cpu().numpy())
            actuals.extend(y_batch.cpu().numpy())
    return mean_absolute_error(actuals, preds)


resultados = {}

# 1. Baseline: Seasonal Naive (7 días)
serie_hist = df_ml["consumo_kwh"]
pred_naive = serie_hist.shift(168).loc[val_df.index].dropna()
actuals_naive = serie_hist.loc[pred_naive.index]
resultados["Seasonal Naive"] = mean_absolute_error(actuals_naive, pred_naive)

# 2. DLinear
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    enable_progress_bar=True,
    logger=False,
    enable_checkpointing=False,
)
modelo_dlinear = DLinear(seq_len=SEQ_LEN, pred_len=1,
                          n_features=len(feature_cols)-1, lr=1e-4)
trainer.fit(modelo_dlinear, train_loader, val_loader)
resultados["DLinear"] = evaluar_modelo(modelo_dlinear, val_loader)

# 3. LSTM (sesión 07)
# modelo_lstm = LSTMForecaster(n_features=3, hidden_size=64)
# trainer.fit(modelo_lstm, train_loader, val_loader)
# resultados["LSTM"] = evaluar_modelo(modelo_lstm, val_loader)

# 4. TCN (sesión 08)
# modelo_tcn = TCNForecaster(n_features=3)
# trainer.fit(modelo_tcn, train_loader, val_loader)
# resultados["TCN"] = evaluar_modelo(modelo_tcn, val_loader)

print("\n=== TABLA COMPARATIVA ===")
for nombre, mae in sorted(resultados.items(), key=lambda x: x[1]):
    print(f"  {nombre:<20s}: MAE = {mae:.4f}")

---

## 7. Actividades de clase

### Actividad 1: El Experimento DLinear vs Naive
Entrena el `DLinear` sobre los datos del taller y compara su MAE en validación con el del Seasonal Naive a 7 días. Si el DLinear pierde, reflexiona: ¿qué nos dice sobre la naturaleza de nuestra serie? ¿Es tan repetitiva que ningún modelo puede superar a "lo de la semana pasada"?

### Actividad 2: Visualizando la Atención
Implementa una versión mínima de *Self-Attention* para una secuencia de ejemplo de 24 horas. Visualiza la **matriz de atención** (un mapa de calor 24×24) donde la intensidad del color representa cuánto presta atención el instante $t$ al instante $t'$. ¿Qué patrones ves? ¿Hay instantes que prestan más atención a las mismas horas del día anterior?

In [ ]:
def self_attention_visual(x):
    """
    x: Tensor [Sequence, Features]
    Devuelve la matriz de atención para visualización.
    """
    Q = K = x  # En el caso más simple, Q=K=V=x
    scores = torch.matmul(Q, K.T) / (x.shape[-1] ** 0.5)
    return torch.softmax(scores, dim=-1)

# Tomar 24 horas de la serie y visualizar
x_ejemplo = torch.tensor(df_ml[feature_cols[:-1]].iloc[:24].values, dtype=torch.float32)
attn = self_attention_visual(x_ejemplo)

plt.figure(figsize=(8, 6))
plt.imshow(attn.numpy(), cmap="viridis", aspect="auto")
plt.colorbar(label="Peso de Atención")
plt.xlabel("Hora (a la que se atiende)")
plt.ylabel("Hora (que atiende)")
plt.title("Matriz de Self-Attention en 24 horas de consumo")
plt.show()

### Actividad 3: El Debate Crítico
Divide la clase en dos grupos. Un grupo defiende: *"Los Transformers son el futuro del forecasting"*. El otro defiende: *"El Seasonal Naive con buen Feature Engineering es suficiente para el 90% de los casos reales de negocio"*. Cada grupo debe aportar argumentos técnicos basados en las sesiones del taller.

---

## 8. Ideas clave

- **La atención es potente, pero tiene un coste $O(L^2)$:** Para series largas, los Transformers son computacionalmente prohibitivos sin modificaciones como el parcheo.
- **Los tokens temporales deben tener semántica:** Un único valor flotante (`1.85 kWh`) no tiene suficiente riqueza informativa para que la atención Q-K sea útil. Los parches (segmentos de la serie) sí la tienen.
- **DLinear nos da una lección de humildad:** Un modelo de dos capas lineales puede superar a arquitecturas con millones de parámetros si los datos tienen una estructura temporal suficientemente regular. La complejidad no es siempre la respuesta.
- **Los Foundation Models cambian la ecuación:** Para casos con pocos datos históricos o dominios muy diversos, un modelo pre-entrenado con billones de puntos puede ser la mejor opción sin necesidad de entrenamiento.
- **El hype científico existe:** La publicación de papers en IA tiene sus propios sesgos. Un modelo que supera a 10 Transformers en 5 benchmarks puede ser inutilizable en un problema de negocio real. Piensa siempre en el contexto.

## Continuación

Con esta sesión cerramos el bloque de arquitecturas de Deep Learning para series temporales. Hemos recorrido el camino completo: desde los modelos locales (Sesiones 01-06) hasta las LSTM, TCNs y el debate sobre los Transformers. Ahora es el momento de demostrar lo aprendido. La **Sesión 10** es el Proyecto Final: un reto integrador donde tendrás que tomar decisiones reales sobre qué modelo usar, cómo fusionar fuentes de datos heterogéneas y cómo evaluar honestamente tus resultados.